In [ ]:
# Ячейка 0 — что делать дальше (просто прочитайте вывод после Run)

from IPython.display import Markdown, display

display(
    Markdown(
        """
### MinerU в Colab

1. В меню: **Runtime → Change runtime type → GPU** (желательно).
2. Ниже **по порядку** выполните ячейки **1 → 5** (кнопка Run на каждой). Ничего в терминале вводить не нужно.
3. После **первой** установки MinerU иногда нужно **Runtime → Restart session**, затем снова ячейки **1–3** и **5**.

**Куда смотреть результат:** папка `output/mineru_benchmark/` в клоне репозитория (текст в `.md`, сводка в `mineru_summaries.json`).
"""
    )
)
print("Готово: выполняйте ячейку 1.")


In [ ]:
# Ячейка 1 — клон репозитория (только Colab) + jiwer

from __future__ import annotations

import os
import shutil
import subprocess
import sys
from pathlib import Path


def in_colab() -> bool:
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


GIT_URL = os.environ.get(
    "OCR_ANALYZE_GIT_URL",
    "https://github.com/developer-mixa/OCR-Analyze.git",
)
REPO_DIR = Path(os.environ.get("OCR_ANALYZE_COLAB_DIR", "/content/OCR-Analyze"))

if in_colab():
    if not (REPO_DIR / "scripts").is_dir():
        print("Клонирую репозиторий…")
        subprocess.check_call(["git", "clone", "--depth", "1", GIT_URL, str(REPO_DIR)])
    os.chdir(REPO_DIR)
    print("Рабочая папка:", Path.cwd().resolve())
    if (REPO_DIR / ".git").is_dir():

        def _pull() -> subprocess.CompletedProcess[str]:
            return subprocess.run(
                ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
                capture_output=True,
                text=True,
            )

        pr = _pull()
        combined = (pr.stderr or "") + (pr.stdout or "")
        if pr.returncode != 0 and "would be overwritten by merge" in combined:
            for sub in ("mineru_benchmark", "paddle_benchmark", "got_benchmark"):
                d = REPO_DIR / "output" / sub
                if d.is_dir():
                    print("Удаляю (мешало git pull):", d)
                    shutil.rmtree(d, ignore_errors=True)
            pr = _pull()
            combined = (pr.stderr or "") + (pr.stdout or "")
        if pr.returncode == 0:
            print("git pull: OK")
        else:
            print("git pull: код", pr.returncode)
            print(combined[:1200] if combined else "(нет вывода)")
else:
    print("Не Colab — откройте ноутбук из корня репозитория. Текущая папка:", Path.cwd().resolve())

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "jiwer"])
print("OK: jiwer. Следующая — ячейка 2 (MinerU).")


In [ ]:
# Ячейка 2 — установка MinerU (долго). Переключатель: только pipeline или полный набор

import os
import shutil
import subprocess
import sys

INSTALL_MINERU_ALL = False  # True = тяжелее, больше пакетов

spec = "mineru[all]" if INSTALL_MINERU_ALL else "mineru[pipeline]"
print("Ставлю", spec, "…")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", spec])
exe = shutil.which("mineru")
print("mineru найден:", exe or "нет в PATH — сделайте Restart runtime и снова ячейки 1–2")


In [ ]:
# Ячейка 3 — пути, список PNG, создаём папку результатов заранее

from __future__ import annotations

from pathlib import Path

REPO_ROOT_OVERRIDE: Path | None = None  # если нужно: Path("/content/мой-клон")


def find_repo_root() -> Path:
    if REPO_ROOT_OVERRIDE is not None:
        p = REPO_ROOT_OVERRIDE.expanduser().resolve()
        if (p / "scripts").is_dir():
            return p
    cwd = Path.cwd().resolve()
    for start in [cwd, *cwd.parents]:
        if (start / "scripts" / "mineru_image_benchmark.py").is_file():
            return start
    return cwd


REPO_ROOT = find_repo_root()
INPUT_DIR = REPO_ROOT / "input" / "data" / "1"
SCRIPT = REPO_ROOT / "scripts" / "mineru_image_benchmark.py"
OUT_DIR = REPO_ROOT / "output" / "mineru_benchmark"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT.resolve())
print("Скрипт есть:", SCRIPT.is_file())
print("Входные PNG:", INPUT_DIR.resolve(), "— папка есть:", INPUT_DIR.is_dir())
print("Результаты будут в:", OUT_DIR.resolve())

_png = sorted(INPUT_DIR.glob("*.png")) if INPUT_DIR.is_dir() else []
print("Найдено PNG:", len(_png))
for p in _png:
    stem = p.stem
    refs = [
        n
        for n in (f"{stem}.ref.txt", f"{stem}.ref.md", f"{stem}.txt", f"{stem}.md")
        if (INPUT_DIR / n).is_file()
    ]
    print(" ", p.name, "| эталон:", ", ".join(refs) if refs else "нет")
if not _png:
    print("Добавьте PNG в input/data/1 внутри клона.")
print("Следующая — ячейка 4 (опция ModelScope) или сразу 5.")


In [ ]:
# Ячейка 4 — опция: если веса с Hugging Face не качаются, поставьте True и выполните эту ячейку

import os

USE_MODELSCOPE = False  # True — зеркало ModelScope вместо HF

if USE_MODELSCOPE:
    os.environ["MINERU_MODEL_SOURCE"] = "modelscope"
    print("Включено: MINERU_MODEL_SOURCE = modelscope")
else:
    print("ModelScope не включён (обычно так и оставляют). Для HF-проблем — поставьте USE_MODELSCOPE = True.")


In [ ]:
# Ячейка 5 — прогон MinerU по всем PNG (без ручных команд в терминале)

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

if "REPO_ROOT" not in globals() or "SCRIPT" not in globals():
    raise RuntimeError("Сначала выполните ячейку 3.")

if not shutil.which("mineru"):
    raise RuntimeError("mineru не найден. Ячейка 2, при необходимости Restart runtime.")

# Настройки прогона (меняйте здесь, без терминала)
BACKEND = "pipeline"
LANG = "cyrillic"
METHOD = None  # для сканов иногда: "ocr"

argv = [
    sys.executable,
    str(SCRIPT),
    "--input-dir",
    str(INPUT_DIR),
    "--output-dir",
    str(OUT_DIR),
    "--backend",
    BACKEND,
    "--lang",
    LANG,
]
if METHOD:
    argv.extend(["--method", METHOD])

print("Запуск MinerU по PNG в", INPUT_DIR)
print("Пишем в", OUT_DIR)
subprocess.check_call(argv, cwd=str(REPO_ROOT))

hyp_dir = OUT_DIR / "hypotheses" / "mineru"
print("\nГотовые тексты (.md):")
mds = sorted(hyp_dir.glob("*.md"))
for p in mds:
    print(" ", p.name, p.stat().st_size, "байт")
if not mds:
    print("  нет .md — откройте output/mineru_benchmark/mineru_runs.jsonl (поле error)")

for name in ("mineru_hypotheses_raw.json", "mineru_hypotheses_concat.txt"):
    fp = OUT_DIR / name
    if fp.is_file():
        print("Сводка сырого текста:", fp.name, "—", fp.stat().st_size, "байт")

summ = OUT_DIR / "mineru_summaries.json"
if summ.is_file():
    print("\n--- сводка метрик (файл", summ.name, ") ---")
    txt = summ.read_text(encoding="utf-8")
    print(txt)
    try:
        outs = json.loads(txt).get("mineru", {}).get("_outputs")
        if outs:
            print("\nПолные пути к файлам (скачать из Colab слева в дереве):")
            for k, v in outs.items():
                print(" ", k, "→", v)
    except json.JSONDecodeError:
        pass

print("\nГотово.")
